# 3-Way Cross Validation (with Feature Reduction Support)
This notebook ensures our model isn't overfitting to a specific timeframe by running across 3 different temporal slices of the dataset.

It's designed to accept a `FEATURE_INDICES` array, ensuring we can evaluate full models or heavily reduced lightweight models (e.g., Top 100 features).

In [1]:
import os
import sys
import gc
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure virtual environment paths
base_dir = os.getcwd()
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages) and site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        break

try:
    from thrember.features import PEFeatureExtractor
    extractor = PEFeatureExtractor()
    NDIM = extractor.dim
    print(f"Loaded PEFeatureExtractor with dimension: {NDIM}")
except ImportError:
    NDIM = 2381
    print(f"Warning: 'thrember' not found. Using default default dimension: {NDIM}")

c:\Users\him\ember2024_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded PEFeatureExtractor with dimension: 2568


### 1. Configuration & Data Setup
Set the paths and define the feature reduction mask (if any).

In [2]:
DATASET_DIR = r"Z:\ember2024_train_data"
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

# Configurable: Set to None for all features, or provide a path to the indices array
INDICES_FILENAME = None  # Set to None to run on all features

if INDICES_FILENAME:
    indices_path = os.path.join(DATASET_DIR, INDICES_FILENAME)
    if os.path.exists(indices_path):
        feature_indices = np.load(indices_path)
        print(f"Feature Reduction Enabled: Loading {len(feature_indices)} features.")
    else:
        print(f"Warning: {indices_path} not found. Falling back to all features.")
        feature_indices = np.arange(NDIM)
else:
    print("Using ALL features.")
    feature_indices = np.arange(NDIM)

num_active_features = len(feature_indices)

# Validate dataset size
file_size = os.path.getsize(X_path)
nrows = file_size // (NDIM * 4)

print(f"Total Rows in Dataset: {nrows}")

# Memory map the raw arrays
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, NDIM))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

Using ALL features.
Total Rows in Dataset: 5252000


### 2. Define Data Loader for Arbitrary Splits
To avoid RAM exhaustion, we create a function to precisely chunk only the segments (and features) of the memory map we need.

In [3]:
def load_data_split(split_ranges, feature_mask):
    """
    Loads specific blocks of rows into a contiguous RAM array.
    split_ranges: list of tuples, e.g., [(0, 10000), (20000, 30000)]
    """
    total_rows = sum(end - start for start, end in split_ranges)
    X_out = np.empty((total_rows, len(feature_mask)), dtype=np.float32)
    y_out = np.empty((total_rows,), dtype=np.int32)
    
    current_idx = 0
    chunk_size = 10000
    
    for start_idx, end_idx in split_ranges:
        for chunk_start in range(start_idx, end_idx, chunk_size):
            chunk_end = min(chunk_start + chunk_size, end_idx)
            rows_to_read = chunk_end - chunk_start
            
            # Read block to RAM, then advanced index the specific features
            temp_X = np.array(X_memmap[chunk_start:chunk_end])
            X_out[current_idx:current_idx + rows_to_read] = temp_X[:, feature_mask]
            y_out[current_idx:current_idx + rows_to_read] = y_memmap[chunk_start:chunk_end]
            
            current_idx += rows_to_read
            
            if current_idx % 500000 == 0 or current_idx == total_rows:
                print(f"  -> Loaded {current_idx} / {total_rows} rows into RAM...")
            
    # Filter out unlabeled samples (-1)
    valid_mask = y_out != -1
    return X_out[valid_mask], y_out[valid_mask]

### 3. Define Cross-Validation Routine
We calculate the indices for:
- Split A (Baseline): First 90% train, Last 10% test
- Split B (Reverse): Last 90% train, First 10% test
- Split C (Middle): First 45% + Last 45% train, Middle 10% test

In [4]:
split_10 = int(nrows * 0.1)
split_20 = int(nrows * 0.2)
split_40 = int(nrows * 0.4)
split_50 = int(nrows * 0.5)
split_60 = int(nrows * 0.6)
split_80 = int(nrows * 0.8)
split_90 = int(nrows * 0.9)

cv_configs = {
    "Split A (Forward 80-10-10)": {
        "train_ranges": [(0, split_80)],
        "valid_ranges": [(split_80, split_90)],
        "test_ranges": [(split_90, nrows)]
    },
    "Split B (Reverse 80-10-10)": {
        "train_ranges": [(split_20, nrows)],
        "valid_ranges": [(split_10, split_20)],
        "test_ranges": [(0, split_10)]
    },
    "Split C (Middle 80-10-10)": {
        "train_ranges": [(0, split_40), (split_60, nrows)],
        "valid_ranges": [(split_40, split_50)],
        "test_ranges": [(split_50, split_60)]
    }
}

params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 1024,
    "max_depth": 15,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1
}

# Fix categorical features index mapping if needed
original_categoricals = [2, 3, 4, 5, 6, 701, 702]
reduced_categoricals = [i for i, orig_idx in enumerate(feature_indices) if orig_idx in original_categoricals]

### 4. Execute Cross-Validation

In [5]:
results = []

for split_name, config in cv_configs.items():
    print(f"\n{'='*50}")
    print(f"🚀 RUNNING: {split_name}")
    print(f"{'='*50}")
    
    # 0. Checkpoint Logic
    final_model_path = os.path.join(DATASET_DIR, f"cv_model_{split_name.replace(' ', '_')[:7]}.txt")
    checkpoint_path = os.path.join(DATASET_DIR, f"cv_checkpoint_{split_name.replace(' ', '_')[:7]}.txt")
    state_file = os.path.join(DATASET_DIR, f"cv_state_{split_name.replace(' ', '_')[:7]}.txt")
    
    if os.path.exists(final_model_path):
        print(f"⏭️ Final model found for {split_name}. Skipping training and loading existing model...")
        model = lgb.Booster(model_file=final_model_path)
        train_duration = 0.0  # Skip tracking duration since it's pre-trained
    else:
        # 1. Train Model in Chunks to prevent RAM crashes
        print("Training incrementally via memory-safe chunks (2 Epochs)...")
        start_time = time.time()
        model = None
        
        chunk_size = 10000  # Smaller chunks to ensure visibility and prevent memory spikes
        epochs = 2          # Make 2 full passes over the dataset
        boost_rounds_per_chunk = 5
        
        # Calculate total chunks primarily for tracking
        total_chunks = sum((end - start + chunk_size - 1) // chunk_size for start, end in config["train_ranges"])
        
        # Resume state setup
        start_epoch = 0
        start_chunk_idx = 0
        if os.path.exists(state_file) and os.path.exists(checkpoint_path):
            with open(state_file, "r") as f:
                parts = f.read().strip().split(",")
                if len(parts) == 2:
                    start_epoch = int(parts[0])
                    start_chunk_idx = int(parts[1])
            model = lgb.Booster(model_file=checkpoint_path)
            print(f"🔄 Checkpoint found! Resuming {split_name} from Epoch {start_epoch + 1}, Chunk {start_chunk_idx}...")
        
        for epoch in range(start_epoch, epochs):
            print(f"\n  --- {split_name}: EPOCH {epoch + 1} / {epochs} ---")
            current_chunk = 0
            
            for start_idx, end_idx in config["train_ranges"]:
                for chunk_start in range(start_idx, end_idx, chunk_size):
                    # Fast-forward to our resume point if we restarted mid-epoch
                    if epoch == start_epoch and current_chunk < start_chunk_idx:
                        current_chunk += 1
                        continue
                        
                    chunk_end = min(chunk_start + chunk_size, end_idx)
                    current_chunk += 1
                    
                    print(f"  [Epoch {epoch+1} | {current_chunk}/{total_chunks}] Training rows {chunk_start} to {chunk_end}...")
                    
                    # Load only this chunk into RAM
                    X_chunk = np.array(X_memmap[chunk_start:chunk_end])
                    y_chunk = np.array(y_memmap[chunk_start:chunk_end])
                    
                    # Apply Feature Mask
                    X_chunk = X_chunk[:, feature_indices]
                    
                    valid_mask = y_chunk != -1
                    X_chunk = X_chunk[valid_mask]
                    y_chunk = y_chunk[valid_mask]
                    
                    if len(y_chunk) == 0:
                        continue
                        
                    train_data = lgb.Dataset(
                        X_chunk, 
                        label=y_chunk, 
                        categorical_feature=reduced_categoricals,
                        free_raw_data=True
                    )
                    
                    # Train 5 Trees per chunk increment (No memory-heavy validation array)
                    model = lgb.train(
                        params,
                        train_data,
                        num_boost_round=boost_rounds_per_chunk,
                        init_model=model,
                        keep_training_booster=True
                    )
                    
                    # Memory Cleanup
                    del train_data
                    del X_chunk
                    del y_chunk
                    gc.collect()
                    
                    # Save a rolling checkpoint every 100 chunks (~1M rows)
                    if current_chunk % 100 == 0:
                        model.save_model(checkpoint_path)
                        with open(state_file, "w") as f:
                            f.write(f"{epoch},{current_chunk}")
                            
            # End of Epoch Checkpoint (Reset chunk counter for next epoch)
            model.save_model(checkpoint_path)
            with open(state_file, "w") as f:
                f.write(f"{epoch + 1},0")
                
        train_duration = time.time() - start_time
        print(f"Training completed in {train_duration:.2f} seconds.")
        
        # Save final model for this specific split and clean up states
        model.save_model(final_model_path)
        if os.path.exists(state_file):
            os.remove(state_file)
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        print(f"💾 Final Model saved to: {final_model_path}")
    
    # 2. Predict & Evaluate entirely in Chunks (Zero RAM Risk)
    print("Evaluating Test Data in memory-safe chunks...")
    y_true_all = []
    y_pred_prob_all = []
    
    test_chunk_size = 50000
    for start_idx, end_idx in config["test_ranges"]:
        for chunk_start in range(start_idx, end_idx, test_chunk_size):
            chunk_end = min(chunk_start + test_chunk_size, end_idx)
            
            X_chunk = np.array(X_memmap[chunk_start:chunk_end])
            y_chunk = np.array(y_memmap[chunk_start:chunk_end])
            
            X_chunk = X_chunk[:, feature_indices]
            
            valid_mask = y_chunk != -1
            X_chunk = X_chunk[valid_mask]
            y_chunk = y_chunk[valid_mask]
            
            if len(y_chunk) == 0:
                continue
                
            preds = model.predict(X_chunk)
            y_true_all.extend(y_chunk)
            y_pred_prob_all.extend(preds)
            
    y_test = np.array(y_true_all)
    y_pred_prob = np.array(y_pred_prob_all)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    auc = roc_auc_score(y_test, y_pred_prob)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"AUC: {auc:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
    
    results.append({
        "Split": split_name,
        "Training Duration (s)": round(train_duration, 2),
        "AUC": round(auc, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4)
    })
    
    # Cleanup Memory
    del y_test
    del model
    gc.collect()
    
print("\n✅ CROSS VALIDATION COMPLETE!")


🚀 RUNNING: Split A (Forward 80-10-10)
⏭️ Final model found for Split A (Forward 80-10-10). Skipping training and loading existing model...
Evaluating Test Data in memory-safe chunks...
AUC: 0.7601 | Acc: 0.7589 | F1: 0.7498

🚀 RUNNING: Split B (Reverse 80-10-10)
Training incrementally via memory-safe chunks (2 Epochs)...
🔄 Checkpoint found! Resuming Split B (Reverse 80-10-10) from Epoch 1, Chunk 100...

  --- Split B (Reverse 80-10-10): EPOCH 1 / 2 ---
  [Epoch 1 | 101/421] Training rows 2050400 to 2060400...
  [Epoch 1 | 102/421] Training rows 2060400 to 2070400...
  [Epoch 1 | 103/421] Training rows 2070400 to 2080400...
  [Epoch 1 | 104/421] Training rows 2080400 to 2090400...
  [Epoch 1 | 105/421] Training rows 2090400 to 2100400...
  [Epoch 1 | 106/421] Training rows 2100400 to 2110400...
  [Epoch 1 | 107/421] Training rows 2110400 to 2120400...
  [Epoch 1 | 108/421] Training rows 2120400 to 2130400...
  [Epoch 1 | 109/421] Training rows 2130400 to 2140400...


KeyboardInterrupt: 

### 5. Final Report & Comparison

In [ ]:
df_results = pd.DataFrame(results)

# Display tabular data
display(df_results)

# Save to CSV
report_path = os.path.join(DATASET_DIR, "cross_validation_report.csv")
df_results.to_csv(report_path, index=False)
print(f"Report saved to: {report_path}")

# Plot AUC comparison
plt.figure(figsize=(8, 5))
sns.barplot(data=df_results, x='Split', y='AUC', palette='viridis')
plt.title('AUC Score Comparison Across Temporal Splits')
plt.ylim(0.8, 1.0)
for index, value in enumerate(df_results['AUC']):
    plt.text(index, value + 0.005, f"{value:.4f}", ha='center')
plt.tight_layout()
plt.show()

# Plot Duration comparison
plt.figure(figsize=(8, 5))
sns.barplot(data=df_results, x='Split', y='Training Duration (s)', palette='magma')
plt.title('Training Duration Comparison')
for index, value in enumerate(df_results['Training Duration (s)']):
    plt.text(index, value + 1, f"{value:.1f}s", ha='center')
plt.tight_layout()
plt.show()